In [50]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 4.3 MB/s eta 0:00:00


In [60]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold,RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import VotingClassifier
import pickle
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report

In [32]:
df_all=pd.read_csv('umap_all_data.csv')
df_umap=pd.read_csv('umap_target_data.csv')

In [53]:
#отделяем признаки от целевой переменной и делаем ее логарифмирование
targets = ['IC50, mM', 'CC50, mM', 'SI']
all_features = [col for col in df_all.columns if col not in targets]
X_all = df_all[all_features].copy()
all_features2 = [col for col in df_umap.columns if col not in targets]
X_umap = df_umap[all_features2].copy()
y_a = df_all['SI']
y_u = df_umap['SI']

In [54]:
#делим выборку на обучающую и валидационную
X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(X_all, y_a, test_size=0.2, random_state=5)
X_u_train, X_u_test, y_u_train, y_u_test = train_test_split(X_umap, y_u, test_size=0.2, random_state=5)


In [55]:
#считаем медиану и делаем разметку классов больше/меньше медианы в train и test выборках
median_a_train = y_a_train.median()
y_a_train_cls = (y_a_train > median_a_train).astype(int)
y_a_test_cls = (y_a_test > median_a_train).astype(int)

median_u_train = y_u_train.median()
y_u_train_cls = (y_u_train > median_u_train).astype(int)
y_u_test_cls = (y_u_test > median_u_train).astype(int)

In [56]:
print('Распределение целевой переменной в обучающей выборке')
print(y_a_train_cls.value_counts())
print('Распределение целевой переменной в валидационной выборке')
print(y_a_test_cls.value_counts())

Распределение целевой переменной в обучающей выборке
SI
1    399
0    399
Name: count, dtype: int64
Распределение целевой переменной в валидационной выборке
SI
0    108
1     92
Name: count, dtype: int64


In [57]:
scaler_robust = RobustScaler()
X_a_train_r = pd.DataFrame(scaler_robust.fit_transform(X_a_train), columns=X_a_train.columns, index=X_a_train.index)
X_a_test_r = pd.DataFrame(scaler_robust.transform(X_a_test), columns=X_a_test.columns, index=X_a_test.index)

In [38]:
#выбираем нелинейные модели для обучения
#определяем cv для кроссвалидации
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=5)
models = {'Random Forest': {'model': RandomForestClassifier(random_state=5, n_jobs=-1),
        'params': {'n_estimators': [200, 300, 350],
                   'max_depth': [10, 20, 30, None],
                   'min_samples_split': [5, 10, 15, 20],
                   'min_samples_leaf': [2, 4, 5]}},
          'Gradient Boosting': {'model': GradientBoostingClassifier(random_state=5),
        'params': {'n_estimators': [200, 300],
                   'learning_rate': [0.01, 0.05, 0.1],
                   'max_depth': [2, 3, 4],
                   'subsample': [0.6, 0.8, 1.0]}},
          'XGBoost': {'model': xgb.XGBClassifier(random_state=5, eval_metric='logloss'),
        'params': {'n_estimators': [200, 250],
            'max_depth': [2, 3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.2]}},
          'LightGBM': {'model': lgb.LGBMClassifier(random_state=5, verbose=-1),
        'params': {'n_estimators': [100, 150],
            'num_leaves': [10, 15, 30],
            'learning_rate': [0.05, 0.1, 0.2]}},
          'SVM': {'model': SVC(random_state=5, probability=True),
        'params': {'C': [1, 10, 30, 50],
                   'gamma': ['scale', 'auto', 0.1, 1],
                   'kernel': ['rbf']}},
          'KNN': {'model': KNeighborsClassifier(),
        'params': {'n_neighbors': [3, 5, 7, 11, 15],
                    'weights': ['uniform', 'distance'],
                   'metric': ['euclidean', 'manhattan']}}}


In [39]:
#обучаем модели и сохраняем лучшие результаты
results = []
best_models = {}
print('Модели на всех признаках')
for name, config in models.items():
  grid = GridSearchCV(config['model'], config['params'],
        cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
  grid.fit(X_a_train_r, y_a_train_cls)

  best_model = grid.best_estimator_
  best_models[name] = best_model

  y_pred = best_model.predict(X_a_test_r)
  y_pred_proba = best_model.predict_proba(X_a_test_r)[:, 1]
  results.append({'Model': name,
        'Best Params': str(grid.best_params_),
        'Accuracy': accuracy_score(y_a_test_cls, y_pred),
        'F1-Score': f1_score(y_a_test_cls, y_pred),
        'ROC-AUC': roc_auc_score(y_a_test_cls, y_pred_proba),
        'CV Score': grid.best_score_})
  print(f'\n{name}')
  print(f'Лучшие параметры: {grid.best_params_}')
  print(f'Accuracy: {accuracy_score(y_a_test_cls, y_pred):.3f}')
  print(f'ROC-AUC:  {roc_auc_score(y_a_test_cls, y_pred_proba):.3f}')
  print(f'F1-Score: {f1_score(y_a_test_cls, y_pred):.3f}')

Модели на всех признаках

Random Forest
Лучшие параметры: {'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 20, 'n_estimators': 200}
Accuracy: 0.710
ROC-AUC:  0.748
F1-Score: 0.663

Gradient Boosting
Лучшие параметры: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.6}
Accuracy: 0.735
ROC-AUC:  0.753
F1-Score: 0.686

XGBoost
Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 250}
Accuracy: 0.710
ROC-AUC:  0.739
F1-Score: 0.663

LightGBM
Лучшие параметры: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 10}
Accuracy: 0.675
ROC-AUC:  0.721
F1-Score: 0.615

SVM
Лучшие параметры: {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
Accuracy: 0.705
ROC-AUC:  0.723
F1-Score: 0.642

KNN
Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'uniform'}
Accuracy: 0.620
ROC-AUC:  0.669
F1-Score: 0.568


In [40]:
results = []
best_models = {}
print('Модели на UMAP признаках')
for name, config in models.items():
  grid = GridSearchCV(config['model'], config['params'],
        cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
  grid.fit(X_u_train, y_u_train_cls)

  best_model = grid.best_estimator_
  best_models[name] = best_model

  y_pred = best_model.predict(X_u_test)
  y_pred_proba = best_model.predict_proba(X_u_test)[:, 1]
  results.append({'Model': name,
        'Best Params': str(grid.best_params_),
        'Accuracy': accuracy_score(y_u_test_cls, y_pred),
        'F1-Score': f1_score(y_u_test_cls, y_pred),
        'ROC-AUC': roc_auc_score(y_u_test_cls, y_pred_proba),
        'CV Score': grid.best_score_})
  print(f'\n{name}')
  print(f'Лучшие параметры: {grid.best_params_}')
  print(f'Accuracy: {accuracy_score(y_u_test_cls, y_pred):.3f}')
  print(f'ROC-AUC:  {roc_auc_score(y_u_test_cls, y_pred_proba):.3f}')
  print(f'F1-Score: {f1_score(y_u_test_cls, y_pred):.3f}')

Модели на UMAP признаках

Random Forest
Лучшие параметры: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 20, 'n_estimators': 300}
Accuracy: 0.630
ROC-AUC:  0.654
F1-Score: 0.560

Gradient Boosting
Лучшие параметры: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.6}
Accuracy: 0.650
ROC-AUC:  0.651
F1-Score: 0.568

XGBoost
Лучшие параметры: {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 200}
Accuracy: 0.645
ROC-AUC:  0.638
F1-Score: 0.559

LightGBM
Лучшие параметры: {'learning_rate': 0.05, 'n_estimators': 150, 'num_leaves': 10}
Accuracy: 0.620
ROC-AUC:  0.610
F1-Score: 0.558

SVM
Лучшие параметры: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
Accuracy: 0.665
ROC-AUC:  0.689
F1-Score: 0.608

KNN
Лучшие параметры: {'metric': 'euclidean', 'n_neighbors': 15, 'weights': 'distance'}
Accuracy: 0.650
ROC-AUC:  0.662
F1-Score: 0.593


В целом модели только на UMAP признаках дают результаты хуже, поэтому продолжим оптимизацию с данными со всеми выбранными признаками

Выбранные модели показывают очень средние результаты по метрикам, выбираем другой алгоритм

In [77]:
#обучаем алгоритм Catboost
catboost_base = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    cat_features=[],
    random_seed=5,
    auto_class_weights='Balanced',
    verbose=0,
    eval_metric='AUC')

catboost_base.fit( X_a_train_r, y_a_train_cls, verbose=False,
    eval_set=(X_a_test_r , y_a_test_cls), plot=False)
y_pred_base = catboost_base.predict(X_a_test_r)
y_pred_proba_base = catboost_base.predict_proba(X_a_test_r)[:, 1]
print(f"Accuracy:  {accuracy_score(y_a_test_cls, y_pred_base):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_a_test_cls, y_pred_proba_base):.4f}")
print(f"F1-Score:  {f1_score(y_a_test_cls, y_pred_base):.4f}")


Accuracy:  0.7200
ROC-AUC:   0.7517
F1-Score:  0.6854


In [73]:
#Catboost показал хороший результат - оптимизируем параметры
cat_params = {
   'iterations': [400, 500],
    'learning_rate': [0.02, 0.03, 0.01],
    'depth': [3, 4, 6],
    'l2_leaf_reg': [2, 3, 4],
    'auto_class_weights': ['Balanced']}
random_cat = RandomizedSearchCV(
    CatBoostClassifier(random_seed=5, verbose=0, eval_metric='AUC'),
    param_distributions=cat_params,
    n_iter=50, cv=cv,scoring='roc_auc',n_jobs=-1,random_state=5, verbose=1)
random_cat.fit(X_a_train_r, y_a_train_cls)
best_cat = random_cat.best_estimator_
y_pred_cat = best_cat.predict(X_a_test_r)
y_pred_proba_cat = best_cat.predict_proba(X_a_test_r)[:, 1]
print(f'Best params: {random_cat.best_params_}')
print(f"Accuracy: {accuracy_score(y_a_test_cls, y_pred_cat):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_a_test_cls, y_pred_proba_cat):.4f}")
print(f"F1-Score: {f1_score(y_a_test_cls, y_pred_cat):.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'learning_rate': 0.02, 'l2_leaf_reg': 3, 'iterations': 400, 'depth': 4, 'border_count': 254, 'auto_class_weights': 'Balanced'}
Accuracy: 0.7050
ROC-AUC:  0.7554
F1-Score: 0.6550


In [74]:
#оптимизация GradientBoosting
gb_params_optimized = {
    'n_estimators': [200, 250, 300, 350, 400, 500],
    'learning_rate': [0.005, 0.007, 0.01, 0.015, 0.02, 0.03],
    'max_depth': [2, 3, 4, 5, 6, 7],
    'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'min_samples_split': [2, 3, 5, 7, 10],
    'min_samples_leaf': [1, 2, 3, 4, 5],
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7, 0.9]}
random_gb = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), gb_params_optimized, n_iter=100, cv=cv, scoring='roc_auc', random_state=42,n_jobs=-1,verbose=1)
random_gb.fit(X_a_train_r, y_a_train_cls)
print(f'Best params: {random_gb.best_params_}')
best_r_gb = random_gb.best_estimator_
y_pred_r_gb= best_r_gb.predict(X_a_test_r)
y_pred_proba_r_gb = best_r_gb.predict_proba(X_a_test_r)[:, 1]

print(f'Accuracy:  {accuracy_score(y_a_test_cls, y_pred_r_gb):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_a_test_cls, y_pred_proba_r_gb):.4f}')
print(f'F1-Score:  {f1_score(y_a_test_cls, y_pred_r_gb):.4f}')

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params: {'subsample': 0.6, 'n_estimators': 250, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 6, 'learning_rate': 0.007}
Accuracy:  0.7100
ROC-AUC:   0.7566
F1-Score:  0.6506


In [81]:
#сохраняем полученную модель
with open('catboost_base_SI.pkl', 'wb') as f:
    pickle.dump(catboost_base, f)
with open('scaler_SI.pkl', 'wb') as f:
    pickle.dump(scaler_robust, f)
with open('median_SI.pkl', 'wb') as f:
    pickle.dump(median_a_train, f)
print('Модель и компоненты сохранены в сессионное хранилище')

Модель и компоненты сохранены в сессионное хранилище


In [82]:
#проверяем загрузку
with open('catboost_base_SI.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('scaler_SI.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
with open('median_SI.pkl', 'rb') as f:
    loaded_median = pickle.load(f)
print('Модель и компоненты загружены')

Модель и компоненты загружены
